In [1]:
from master_llm import MasterLLM
from master_tokenizer import MasterTokenizer
import torch

In [28]:
u_tokenizer = MasterTokenizer("tokenizer.json")

prompt = "the capital of united states and the capital of france"

tokens = u_tokenizer.encode(prompt)

In [29]:
torch.manual_seed(1)

u_model = MasterLLM(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings = u_model(tokens)

In [30]:
u_model

MasterLLM(
  (embedding): Embedding(64, 4)
  (pos_embedding): Embedding(32, 4)
)

In [33]:
from plot_tokens import plot_tokens

sentences = [
    {
        "words": sentence_meanings.detach().numpy(),
        "labels": u_tokenizer.tokenize(prompt),
        "color": "red", 
    }
]

In [35]:
plot_tokens(sentences, "Models Context Space")

In [8]:
# Pozisyon değerleri
the_pos = [-1.5256, -0.7502, -0.6540, -1.6095]
capital_pos = [0.9326, -0.2774,  0.3166, -1.6980]

In [9]:
# Farkların hesaplanması Manhattan distance ile
hardness_dist   = abs(the_pos[0] - capital_pos[0])
brightness_dist = abs(the_pos[1] - capital_pos[1])
redness_dist    = abs(the_pos[2] - capital_pos[2])
blueness_dist   = abs(the_pos[3] - capital_pos[3])

(hardness_dist, brightness_dist, redness_dist, blueness_dist)

(2.4582, 0.4728, 0.9706, 0.08850000000000002)

In [10]:
total_distance = (hardness_dist + brightness_dist + redness_dist  + blueness_dist) 
total_distance

3.9901

In [36]:
apple = [-1.5256, -0.7502, -0.6540, -1.6095]
real_apple = [0.5, -0.7502, -0.6540, -1.6095]

In [37]:
# Farkların hesaplanması kosinüs benzerliği ile
from torch import dist

def is_apple(pos , real_pos):
    dist1 = pos[0] - real_pos[0]

    print("Distance from real apple:", dist1)

    if dist1 > 0:
        apple[0] -= 0.5
    
    else:
        apple[0] += 0.5

    return dist1 > 0 and dist1 < 0.5

is_apple(apple , real_apple)

Distance from real apple: -2.0256


False

In [14]:
cos_sim_hardness = the_pos[0]*capital_pos[0] 
cos_sim_brightness = the_pos[1]*capital_pos[1] 
cos_sim_redness = the_pos[3]*capital_pos[3]

cos_sim_hardness, cos_sim_brightness, cos_sim_redness

total_distance = cos_sim_hardness + cos_sim_brightness + cos_sim_redness
total_distance

1.5182619199999996

In [15]:
sentence_meanings[0],sentence_meanings[1],sentence_meanings[2],sentence_meanings[3]

(tensor([-1.5256, -0.7502, -0.6540, -1.6095], grad_fn=<SelectBackward0>),
 tensor([-0.2092, -0.2013, -0.7509, -0.0189], grad_fn=<SelectBackward0>),
 tensor([ 0.9326, -0.2774,  0.3166, -1.6980], grad_fn=<SelectBackward0>),
 tensor([ 0.7698, -0.1935,  0.1223, -0.0585], grad_fn=<SelectBackward0>))

In [38]:
cs_0_0 = sentence_meanings[0][0] * sentence_meanings[0][0] + sentence_meanings[0][1] * sentence_meanings[0][1] + sentence_meanings[0][2] * sentence_meanings[0][2] + sentence_meanings[0][3] * sentence_meanings[0][3]
cs_0_1 = sentence_meanings[0][0] * sentence_meanings[1][0] + sentence_meanings[0][1] * sentence_meanings[1][1] + sentence_meanings[0][2] * sentence_meanings[1][2] + sentence_meanings[0][3] * sentence_meanings[1][3]
cs_0_2 = sentence_meanings[0][0] * sentence_meanings[2][0] + sentence_meanings[0][1] * sentence_meanings[2][1] + sentence_meanings[0][2] * sentence_meanings[2][2] + sentence_meanings[0][3] * sentence_meanings[2][3]
cs_0_3 = sentence_meanings[0][0] * sentence_meanings[3][0] + sentence_meanings[0][1] * sentence_meanings[3][1] + sentence_meanings[0][2] * sentence_meanings[3][2] + sentence_meanings[0][3] * sentence_meanings[3][3]

cs_0_0, cs_0_1, cs_0_2, cs_0_3

(tensor(5.9084, grad_fn=<AddBackward0>),
 tensor(0.9915, grad_fn=<AddBackward0>),
 tensor(1.3112, grad_fn=<AddBackward0>),
 tensor(-1.0151, grad_fn=<AddBackward0>))

In [39]:
the_similarities = []

for i in range(sentence_meanings.shape[0]):
  cs_the_i = sentence_meanings[0][0] * sentence_meanings[i][0] + sentence_meanings[0][1] * sentence_meanings[i][1] + sentence_meanings[0][2] * sentence_meanings[i][2] + sentence_meanings[0][3] * sentence_meanings[i][3]
  the_similarities.append(cs_the_i)

the_similarities

[tensor(5.9084, grad_fn=<AddBackward0>),
 tensor(0.9915, grad_fn=<AddBackward0>),
 tensor(1.3112, grad_fn=<AddBackward0>),
 tensor(-1.0151, grad_fn=<AddBackward0>),
 tensor(-0.5984, grad_fn=<AddBackward0>),
 tensor(0.5215, grad_fn=<AddBackward0>),
 tensor(-2.9568, grad_fn=<AddBackward0>),
 tensor(1.3844, grad_fn=<AddBackward0>),
 tensor(3.2495, grad_fn=<AddBackward0>),
 tensor(1.0536, grad_fn=<AddBackward0>),
 tensor(-0.7985, grad_fn=<AddBackward0>),
 tensor(-0.3483, grad_fn=<AddBackward0>),
 tensor(1.3706, grad_fn=<AddBackward0>),
 tensor(3.3436, grad_fn=<AddBackward0>),
 tensor(0.6584, grad_fn=<AddBackward0>),
 tensor(-1.5429, grad_fn=<AddBackward0>),
 tensor(-0.9496, grad_fn=<AddBackward0>),
 tensor(-0.4411, grad_fn=<AddBackward0>),
 tensor(1.0307, grad_fn=<AddBackward0>),
 tensor(-3.9441, grad_fn=<AddBackward0>)]

In [40]:
all_similarities = torch.zeros((sentence_meanings.shape[0], sentence_meanings.shape[0]))

for j in range(sentence_meanings.shape[0]):
   j_similarities= torch.zeros(sentence_meanings.shape[0])

   for i in range(sentence_meanings.shape[0]):
        for k in range(sentence_meanings.shape[1]):
         cs_j_i = sentence_meanings[j][k]*sentence_meanings[i][k] 
         j_similarities[i]= cs_j_i
         
   all_similarities[j]=j_similarities

all_similarities.detach().numpy()


array([[ 2.59044123e+00,  3.03506367e-02,  2.73298335e+00,
         9.41060483e-02,  1.82404041e-01,  1.54109746e-01,
        -2.15441155e+00,  2.07969561e-01,  1.11694682e+00,
         9.53758895e-01,  2.72630781e-01, -3.98878753e-01,
         3.02480131e-01,  1.85642338e+00,  3.20270538e-01,
         1.16122055e+00,  3.25292796e-01, -5.36906481e-01,
         3.17346632e-01, -2.92590767e-01],
       [ 3.03506367e-02,  3.55600088e-04,  3.20207179e-02,
         1.10258372e-03,  2.13711802e-03,  1.80561084e-03,
        -2.52419394e-02,  2.43665371e-03,  1.30865909e-02,
         1.11746173e-02,  3.19425040e-03, -4.67342185e-03,
         3.54397693e-03,  2.17505917e-02,  3.75241670e-03,
         1.36053199e-02,  3.81125929e-03, -6.29060948e-03,
         3.71815893e-03, -3.42810946e-03],
       [ 2.73298335e+00,  3.20207179e-02,  2.88336897e+00,
         9.92843434e-02,  1.92441031e-01,  1.62589818e-01,
        -2.27296066e+00,  2.19413325e-01,  1.17840803e+00,
         1.00624061e+00,  2.8

In [18]:
all_similarities.shape

torch.Size([20, 20])

In [19]:
sentence_meanings.shape,sentence_meanings.T.shape

(torch.Size([20, 4]), torch.Size([4, 20]))

In [ ]:
#Query,Key,Value
all_similarities = sentence_meanings @ sentence_meanings.T #value
all_similarities.detach().numpy()

all_similarities.shape

torch.Size([20, 20])

In [21]:
torch.sum(all_similarities[0])

tensor(8.2288, grad_fn=<SumBackward0>)

In [22]:
attention_weights = torch.softmax(all_similarities, dim=1)
attention_weights

tensor([[8.1946e-01, 5.9999e-03, 8.2603e-03, 8.0667e-04, 1.2237e-03, 3.7499e-03,
         1.1572e-04, 8.8869e-03, 5.7380e-02, 6.3839e-03, 1.0018e-03, 1.5713e-03,
         8.7658e-03, 6.3046e-02, 4.3000e-03, 4.7584e-04, 8.6128e-04, 1.4320e-03,
         6.2396e-03, 4.3116e-05],
        [1.1473e-01, 8.1405e-02, 3.0146e-02, 3.4407e-02, 1.8480e-02, 2.9712e-02,
         2.7623e-02, 7.8894e-02, 1.0706e-01, 6.9522e-02, 2.5101e-02, 3.9047e-02,
         4.3478e-02, 1.0564e-01, 7.4684e-02, 1.6299e-02, 2.6907e-02, 2.3175e-02,
         3.5823e-02, 1.7872e-02],
        [4.4356e-02, 8.4658e-03, 6.0869e-01, 2.9684e-02, 1.6212e-02, 1.2135e-02,
         8.5615e-04, 8.5918e-03, 2.2690e-02, 1.2764e-02, 3.1655e-02, 6.7095e-03,
         8.8897e-03, 1.4005e-02, 1.4916e-02, 6.3631e-02, 3.5306e-02, 5.0865e-03,
         1.0105e-02, 4.5252e-02],
        [1.6006e-02, 3.5703e-02, 1.0968e-01, 8.4471e-02, 4.2736e-02, 3.5703e-02,
         3.0284e-02, 3.0832e-02, 3.1857e-02, 2.4954e-02, 7.2045e-02, 3.7881e-02,
       

In [23]:
torch.sum(attention_weights[0])

tensor(1.0000, grad_fn=<SumBackward0>)

In [24]:
sentence_context_vectors = attention_weights @ sentence_meanings
sentence_context_vectors,sentence_meanings

(tensor([[-1.3642, -0.5804, -0.6650, -1.4544],
         [-0.3833,  0.0771, -0.3690, -0.4858],
         [ 0.6289, -0.0605,  0.2308, -1.2028],
         [ 0.4327,  0.2193,  0.2051, -0.2858],
         [ 0.0623,  0.3735,  0.4016, -0.1809],
         [-0.2149,  0.1034,  0.1941, -0.2750],
         [-0.0276,  0.8163,  0.2735,  0.6393],
         [-0.4880,  0.0775, -0.4011, -0.5648],
         [-0.8795, -0.4359, -0.5518, -1.0064],
         [-0.6646,  1.0497, -0.8285, -0.8352],
         [ 0.3295,  0.2349,  0.3010, -0.2964],
         [-0.1033,  0.2790,  0.0843, -0.1576],
         [-0.4629,  0.0648, -0.0904, -0.4583],
         [-0.9981,  1.1521, -1.1039, -1.0766],
         [-0.2338,  0.1810, -0.3295, -0.4863],
         [ 0.6268,  1.0575,  0.4230, -0.4987],
         [ 0.4249,  0.2853,  0.2814, -0.3113],
         [-0.1742,  0.2892,  0.3568, -0.0203],
         [-0.3606,  0.1359,  0.0309, -0.3752],
         [ 1.3265,  0.7178,  0.4320, -0.0393]], grad_fn=<MmBackward0>),
 tensor([[-1.5256, -0.7502, -0.6540

In [25]:
sentence_meanings_without_pos = u_model.embedding(tokens)

In [26]:
sentences = [
    {
        "words": sentence_meanings_without_pos.detach().numpy(),
        "labels": u_tokenizer.tokenize(prompt),
        "color": "blue",
    },
    {
        "words": sentence_meanings.detach().numpy(),
        "labels": u_tokenizer.tokenize(prompt),
        "color": "purple", 
    },
    {
        "words": sentence_context_vectors.detach().numpy(),
        "labels": u_tokenizer.tokenize(prompt),
        "color": "orange", 
    }
]
plot_tokens(sentences, "Models Attention Sentence Space")